In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *
from pyspark.sql.window import *

### Scenario:
You're ingesting `order_items_nested.json`, a raw event feed from an order management system. Each order arrives as a single JSON record with a **nested array** of line items — this is a very common real-world shape (event logs, API payloads, NoSQL exports), and it needs to be flattened before it's usable in downstream analytics tables.

**Problem:**

- Read the JSON file (Spark will infer the nested `items` array structure — that's fine for this one, since nested schemas are painful to hand-write, but call `.printSchema()` first and look at how the `items` array is represented before writing any transformation code).
- **Explode** the `items` array so each line item becomes its own row (order-level columns like `order_id`, `customer_id`, `order_date` should repeat across each item row).
- Compute `line_total` = `qty` × `price` for each item.
- Aggregate back up to **one row per order**: `total_order_value` = sum of `line_total` for that order.
- Order the output by `order_id` ascending.

**Expected Output**

| order_id | customer_id | order_date | total_order_value |
| :--- | :--- | :--- | :--- |
| O1001 | C001 | 2024-07-01 | 45.0 |
| O1002 | C002 | 2024-07-01 | 15.0 |
| O1003 | C001 | 2024-07-02 | 65.0 |

In [0]:
order_json_df = spark.read.format("json").load(
    "/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/order_items_nested.json"
)

# order_json_df.printSchema()
"""
Json Schema
root
 |-- customer_id: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
"""
orders_df = order_json_df.select(
    "order_id", "customer_id", "order_date", explode("items").alias("order_items")
).select(
    "order_id",
    "customer_id",
    "order_date",
    "order_items.product",
    "order_items.price",
    "order_items.qty",
)

orders_transform_df = orders_df.withColumn("line_total", col("price") * col("qty")).groupBy(
    col("order_id"), col("customer_id"), col("order_date")
).agg(sum(col("line_total")).alias("total_order_value"))

orders_transform_df.orderBy(col("order_id").asc()).show()

### Scenario:
You're given `user_clickstream.csv`, raw click events for users on a website. Product analytics wants events grouped into **sessions** — a classic "gaps and islands" problem. A new session starts whenever a user is inactive for more than 30 minutes between consecutive events.

**Problem:**

- Read the file with an explicit schema.
- For each user, order events by `event_time` and compute the time gap (in minutes) between each event and the previous event for that same user (use `lag()` over a window).
- Flag a row as the start of a new session whenever the gap is greater than 30 minutes, or it's the user's very first event.
- Using a running/cumulative sum of that flag (per user), assign a `session_id` (starting at 1 for each user's first session).
- Group by `user_id` and `session_id` to compute:
  - `event_count` — number of events in that session.
  - `session_start` — earliest `event_time` in the session.
  - `session_end` — latest `event_time` in the session.
  - `duration_minutes` — difference between `session_end` and `session_start`, in minutes.
- Order the output by `user_id` ascending, then `session_id` ascending.

**Schema**

| Column | Type |
| :--- | :--- |
| **user_id** | string |
| **event_time** | timestamp |

**Expected Output**

| user_id | session_id | event_count | session_start | session_end | duration_minutes |
| :--- | :--- | :--- | :--- | :--- | :--- |
| U1 | 1 | 3 | 2024-08-01 09:00:00 | 2024-08-01 09:10:00 | 10 |
| U1 | 2 | 2 | 2024-08-01 09:50:00 | 2024-08-01 09:55:00 | 5 |
| U2 | 1 | 2 | 2024-08-01 10:00:00 | 2024-08-01 10:20:00 | 20 |
| U2 | 2 | 1 | 2024-08-01 11:05:00 | 2024-08-01 11:05:00 | 0 |

In [0]:
schema = StructType(
    [
        StructField("user_id", StringType()),
        StructField("event_time", TimestampType())
    ]
)

user_click_strm_df = spark.read.format("csv").option("header", True).schema(schema).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/user_clickstream.csv")

win_time_gap = Window.partitionBy(col("user_id")).orderBy(col("event_time"))

users_session_flag_df = user_click_strm_df.withColumn(
    "time_gap", (unix_timestamp(col("event_time")) - unix_timestamp(lag(col("event_time")).over(win_time_gap)))/60
).withColumn(
    "session_flag",
    when((col("time_gap").isNull()) | (col("time_gap")>30), lit(1))
    .otherwise(lit(0))
)

users_with_sessions_df = users_session_flag_df.withColumn(
    "session_id",
    sum(col("session_flag")).over(Window.partitionBy(col("user_id")).orderBy(col("event_time")).rowsBetween(Window.unboundedPreceding, Window.currentRow))
)

user_session_summary_df = users_with_sessions_df.groupBy(col("user_id"), col("session_id")).agg(
    count(col("session_flag")).alias("event_count"),
    min(col("event_time")).alias("session_start"),
    max(col("event_time")).alias("session_end")
).withColumn(
    "duration_minutes",
    (unix_timestamp(col("session_end")) - unix_timestamp(col("session_start")))/60
)

user_session_summary_df.orderBy(col("user_id"), col("session_id")).show()